# Лабораторная работа: HR-датасет сотрудников — подготовка данных к анализу

**Курс:** Data Science & Analytics — Неделя 4: Подготовка данных к анализу

**Ситуация.** Я аналитик данных и работаю с HR-отделом. HR-менеджер попросил подготовить датасет сотрудников
к анализу. В исходных данных есть пропуски, дубликаты, несогласованные записи, неверные типы данных
и даты, сохранённые как текст.

**Цель.** Очистить и подготовить HR-датасет с помощью Pandas: найти проблемы качества данных и решить их
подходящими методами и параметрами.

**Правило.** Исходные датасеты (`employees`, `departments`) не изменяются — вся работа ведётся с копией (`df`).

## Подготовка — исходные датасеты (даны в задании, не изменяются)

In [1]:
import pandas as pd
import numpy as np

# Do not modify this cell.
# Raw HR employee dataset created for data cleaning practice.
employees = pd.DataFrame({
    "EmployeeID": [
        101, 102, 103, 103, 105, 106, 107, 108,
        109, 110, 111, 112, 113, 114, 115
    ],
    "FirstName": [
        "Ainura", "Bakyt", "Gulnara", "Gulnara", "Nurlan",
        "Aigul", "Beksultan", "Aisuluu", "Ermek", "Nurgul",
        "Daniyar", "Aizhan", "Kubat", "Perizat", "Adilet"
    ],
    "LastName": [
        "Asanova", "Kyzylov", "Mamatova", "Mamatova", "Abdyldaev",
        "Sultanova", "Toktogulov", "baisalova", "Kadyrov",
        "Jumabaeva ", "Turgunov", "Ryskulova", "Kadyraliev",
        "BAktybekova", "MOLDOKULOV"
    ],
    "Department": [
        "IT", "HR", "it", "IT", "Finance", "HR",
        " IT ", "Finance", "IT", "HR", "it",
        "Financing", "Information Tecnology", "HR", "finance"
    ],
    "Gender": [
        "F", "m", "F", "f", "M", "F", "M", "F",
        "M", "F", "M", "F", "M", "F", "m"
    ],
    "Age": [
        "28", "35", "29", "29", "150", "31", "33",
        "27", "30", "38", "26", "34", "29", "32", "31"
    ],
    "Salary": [
        "80000", "65000", np.nan, np.nan, "90000", "70000",
        "unknown", "62000", "85000", "58000", "72000",
        "66000", "78000", "88000", "61000"
    ],
    "HireDate": [
        "2023-01-15", "2022-06-10", "2024-03-01", "2024-03-01",
        "2021-11-20", "2023-08-05", "2020-05-12", "2022-09-18",
        "2021-03-25", "2019-07-14", "2023-04-18", "2020-10-02",
        "2022-12-11", "2021-08-30", "2023-06-22"
    ],
    "PerformanceScore": [
        4.2, 3.8, 4.5, 4.5, 4.0, 3.9, 4.1,
        3.7, 4.3, 3.6, 4.4, 3.8, 4.2, "4.0", 3.9
    ]
})

departments = pd.DataFrame({
    "DepartmentID": [1, 2, 3],
    "DepartmentName": ["IT", "HR", "Finance"],
    "Location": ["Bishkek", "Osh", "Bishkek"]
})

In [2]:
# Работаем с копией, чтобы исходный датасет остался без изменений
df = employees.copy()
display(df.head())

,EmployeeID,FirstName,LastName,Department,Gender,Age,Salary,HireDate,PerformanceScore
0,101,Ainura,Asanova,IT,F,28,80000,2023-01-15,4.2
1,102,Bakyt,Kyzylov,HR,m,35,65000,2022-06-10,3.8
2,103,Gulnara,Mamatova,it,F,29,NaN,2024-03-01,4.5
3,103,Gulnara,Mamatova,IT,f,29,NaN,2024-03-01,4.5
4,105,Nurlan,Abdyldaev,Finance,M,150,90000,2021-11-20,4.0


## Задание 1 — Осмотр датасета

In [3]:
# Количество строк и столбцов
print(df.shape)

(15, 9)


In [4]:
# Названия столбцов, типы данных и количество заполненных значений
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   EmployeeID        15 non-null     int64 
 1   FirstName         15 non-null     str   
 2   LastName          15 non-null     str   
 3   Department        15 non-null     str   
 4   Gender            15 non-null     str   
 5   Age               15 non-null     str   
 6   Salary            13 non-null     str   
 7   HireDate          15 non-null     str   
 8   PerformanceScore  15 non-null     object
dtypes: int64(1), object(1), str(7)
memory usage: 1.2+ KB


In [5]:
# Типы данных всех столбцов
print(df.dtypes)

EmployeeID           int64
FirstName              str
LastName               str
Department             str
Gender                 str
Age                    str
Salary                 str
HireDate               str
PerformanceScore    object
dtype: object


In [6]:
# Типы данных выбранных столбцов
print(df[["Salary", "Age", "HireDate"]].dtypes)

Salary      str
Age         str
HireDate    str
dtype: object


In [7]:
# notna() -> True там, где зарплата указана
print(df["Salary"].notna())

0      True
1      True
2     False
3     False
4      True
5      True
6      True
7      True
8      True
9      True
10     True
11     True
12     True
13     True
14     True
Name: Salary, dtype: bool


In [8]:
# Строки, где зарплата указана
display(df[df["Salary"].notna()])

,EmployeeID,FirstName,LastName,Department,Gender,Age,Salary,HireDate,PerformanceScore
0,101,Ainura,Asanova,IT,F,28,80000,2023-01-15,4.2
1,102,Bakyt,Kyzylov,HR,m,35,65000,2022-06-10,3.8
4,105,Nurlan,Abdyldaev,Finance,M,150,90000,2021-11-20,4.0
5,106,Aigul,Sultanova,HR,F,31,70000,2023-08-05,3.9
6,107,Beksultan,Toktogulov,IT,M,33,unknown,2020-05-12,4.1
7,108,Aisuluu,baisalova,Finance,F,27,62000,2022-09-18,3.7
8,109,Ermek,Kadyrov,IT,M,30,85000,2021-03-25,4.3
9,110,Nurgul,Jumabaeva,HR,F,38,58000,2019-07-14,3.6
10,111,Daniyar,Turgunov,it,M,26,72000,2023-04-18,4.4
11,112,Aizhan,Ryskulova,Financing,F,34,66000,2020-10-02,3.8


### Поиск проблем качества данных

In [9]:
# 1) Пропуски в каждом столбце
print(df.isnull().sum())

EmployeeID          0
FirstName           0
LastName            0
Department          0
Gender              0
Age                 0
Salary              2
HireDate            0
PerformanceScore    0
dtype: int64

In [10]:
# Текст "unknown" — скрытый пропуск: isnull() его не видит
display(df[df["Salary"] == "unknown"])

,EmployeeID,FirstName,LastName,Department,Gender,Age,Salary,HireDate,PerformanceScore
6,107,Beksultan,Toktogulov,IT,M,33,unknown,2020-05-12,4.1


In [11]:
# 2) Дубликаты
print("Полных дубликатов:", df.duplicated().sum())
display(df[df.duplicated(subset=["EmployeeID"], keep=False)])

Полных дубликатов: 0


,EmployeeID,FirstName,LastName,Department,Gender,Age,Salary,HireDate,PerformanceScore
2,103,Gulnara,Mamatova,it,F,29,NaN,2024-03-01,4.5
3,103,Gulnara,Mamatova,IT,f,29,NaN,2024-03-01,4.5


In [12]:
# 3) Несогласованные названия отделов, коды пола и фамилии
print(df["Department"].unique())
print(df["Gender"].unique())
print(df["LastName"].unique())

<StringArray>
[                   'IT',                    'HR',                    'it',
               'Finance',                  ' IT ',             'Financing',
 'Information Tecnology',               'finance']
Length: 8, dtype: str
<StringArray>
['F', 'm', 'f', 'M']
Length: 4, dtype: str
<StringArray>
[    'Asanova',     'Kyzylov',    'Mamatova',   'Abdyldaev',   'Sultanova',
  'Toktogulov',   'baisalova',     'Kadyrov',  'Jumabaeva ',    'Turgunov',
   'Ryskulova',  'Kadyraliev', 'BAktybekova',  'MOLDOKULOV']
Length: 14, dtype: str


In [13]:
# 4) Неверный возраст: Age хранится как текст, поэтому временно переводим его в число
age_check = pd.to_numeric(df["Age"], errors="coerce")
display(age_check.describe())
display(df[(age_check < 18) | (age_check > 70)])

count     15.000000
mean      38.800000
std       30.926641
min       26.000000
25%       29.000000
50%       31.000000
75%       33.500000
max      150.000000
Name: Age, dtype: float64

,EmployeeID,FirstName,LastName,Department,Gender,Age,Salary,HireDate,PerformanceScore
4,105,Nurlan,Abdyldaev,Finance,M,150,90000,2021-11-20,4.0


In [14]:
# 5) Даты, сохранённые как текст
print(df["HireDate"].dtype)
print(type(df["HireDate"][0]))

str
<class 'str'>


In [15]:
# Лишние пробелы: значение отличается от себя же после str.strip()
text_columns = ["FirstName", "LastName", "Department", "Gender"]

for col in text_columns:
    print(col, df[df[col] != df[col].str.strip()][col].tolist())

FirstName []
LastName ['Jumabaeva ']
Department [' IT ']
Gender []


In [16]:
# Удаляем лишние пробелы в начале и в конце строк
for col in text_columns:
    df[col] = df[col].str.strip()

# Приводим записи к единому виду: фамилии с заглавной буквы, пол — заглавными буквами
df["LastName"] = df["LastName"].str.title()
df["Gender"] = df["Gender"].str.upper()

print(df["Department"].unique())
print(df["Gender"].unique())
print(df["LastName"].unique())

<StringArray>
['IT', 'HR', 'it', 'Finance', 'Financing', 'Information Tecnology', 'finance']
Length: 7, dtype: str
<StringArray>
['F', 'M']
Length: 2, dtype: str
<StringArray>
[    'Asanova',     'Kyzylov',    'Mamatova',   'Abdyldaev',   'Sultanova',
  'Toktogulov',   'Baisalova',     'Kadyrov',   'Jumabaeva',    'Turgunov',
   'Ryskulova',  'Kadyraliev', 'Baktybekova',  'Moldokulov']
Length: 14, dtype: str


In [17]:
# Все ли значения PerformanceScore числовые? Смотрим тип каждого значения
print(df["PerformanceScore"].dtype)
print(df["PerformanceScore"].apply(type).value_counts())

object
PerformanceScore
<class 'float'>    14
<class 'str'>       1
Name: count, dtype: int64


In [18]:
# Значение, сохранённое как текст
display(df[df["PerformanceScore"].apply(type) == str][["EmployeeID", "PerformanceScore"]])

,EmployeeID,PerformanceScore
13,114,4.0


### Задание 1 — Пояснение

- **Размер:** `shape` показывает, что в исходном датасете **15 строк и 9 столбцов**. `info()` дополнительно показывает
  типы данных и количество заполненных значений: у `Salary` только 13 non-null из 15.
- **Типы данных:** `Age`, `Salary` и `HireDate` хранятся как **текст** (`object`/`str`), а не как числа или даты,
  поэтому по ним нельзя посчитать среднее или работать со временем. `PerformanceScore` имеет тип `object`, потому что
  в нём вместе с числами есть строка `"4.0"`.
- **`notna()`** возвращает `True`, где значение есть: зарплата указана у 13 строк, но одно из этих значений —
  слово `"unknown"`, то есть *скрытый* пропуск, который `isnull()` не видит.

**Найденные проблемы качества данных:**

| Проблема | Где |
|---|---|
| Пропуски | `Salary`: 2 × `NaN` + 1 × `"unknown"` |
| Дубликаты | `EmployeeID` 103 (Gulnara Mamatova) встречается дважды |
| Несогласованные названия отделов | `"it"`, `" IT "`, `"Information Tecnology"`, `"Financing"`, `"finance"` |
| Неверный возраст | у Nurlan Abdyldaev (ID 105) `Age = 150` |
| Даты в виде текста | `HireDate` — строковый столбец |
| Лишние пробелы | `" IT "` в Department, `"Jumabaeva "` в LastName |
| Другие несогласованности | коды пола `m`/`f`, фамилии `baisalova`, `BAktybekova`, `MOLDOKULOV` |
| Нечисловая оценка | `PerformanceScore = "4.0"` (текст) у ID 114 |

Две строки с ID 103 пока **не являются** полными дубликатами (`duplicated().sum()` = 0) — они отличаются только
`"it"`/`"IT"` и `"F"`/`"f"`. Поэтому текст нужно очистить *до* удаления дубликатов.

**Очистка в этом задании:** лишние пробелы удалены с помощью `str.strip()`, фамилии приведены к виду
«с заглавной буквы» (`str.title()`), коды пола — к заглавным буквам (`str.upper()`).

## Задание 2 — Обработка пропущенных зарплат

In [19]:
# Количество пропусков в каждом столбце
print(df.isnull().sum())

EmployeeID          0
FirstName           0
LastName            0
Department          0
Gender              0
Age                 0
Salary              2
HireDate            0
PerformanceScore    0
dtype: int64


In [20]:
# Процент пропусков: True = 1, False = 0, поэтому mean() даёт долю пропусков
missing_percent = df.isnull().mean() * 100
print(missing_percent)

EmployeeID           0.000000
FirstName            0.000000
LastName             0.000000
Department           0.000000
Gender               0.000000
Age                  0.000000
Salary              13.333333
HireDate             0.000000
PerformanceScore     0.000000
dtype: float64


In [21]:
# Заменяем текст "unknown" на настоящий пропуск
df["Salary"] = df["Salary"].replace("unknown", np.nan)

# Переводим Salary в число; errors="coerce" превращает неверные значения в NaN
df["Salary"] = pd.to_numeric(df["Salary"], errors="coerce")

print(df["Salary"].dtype)
print(df["Salary"].isnull().sum())

float64
3


In [22]:
# Процент пропусков после замены "unknown"
missing_percent = df.isnull().mean() * 100
print(missing_percent)

EmployeeID           0.0
FirstName            0.0
LastName             0.0
Department           0.0
Gender               0.0
Age                  0.0
Salary              20.0
HireDate             0.0
PerformanceScore     0.0
dtype: float64

### Эксперименты с `dropna()` (результат не сохраняется в `df`)

In [23]:
# axis=0 (по умолчанию), how='any': удаляются строки, где есть хотя бы один пропуск
display(df.dropna())

,EmployeeID,FirstName,LastName,Department,Gender,Age,Salary,HireDate,PerformanceScore
0,101,Ainura,Asanova,IT,F,28,80000.0,2023-01-15,4.2
1,102,Bakyt,Kyzylov,HR,M,35,65000.0,2022-06-10,3.8
4,105,Nurlan,Abdyldaev,Finance,M,150,90000.0,2021-11-20,4.0
5,106,Aigul,Sultanova,HR,F,31,70000.0,2023-08-05,3.9
7,108,Aisuluu,Baisalova,Finance,F,27,62000.0,2022-09-18,3.7
8,109,Ermek,Kadyrov,IT,M,30,85000.0,2021-03-25,4.3
9,110,Nurgul,Jumabaeva,HR,F,38,58000.0,2019-07-14,3.6
10,111,Daniyar,Turgunov,it,M,26,72000.0,2023-04-18,4.4
11,112,Aizhan,Ryskulova,Financing,F,34,66000.0,2020-10-02,3.8
12,113,Kubat,Kadyraliev,Information Tecnology,M,29,78000.0,2022-12-11,4.2


In [24]:
# axis=1: удаляются столбцы, где есть пропуски
display(df.dropna(axis=1).head())

,EmployeeID,FirstName,LastName,Department,Gender,Age,HireDate,PerformanceScore
0,101,Ainura,Asanova,IT,F,28,2023-01-15,4.2
1,102,Bakyt,Kyzylov,HR,M,35,2022-06-10,3.8
2,103,Gulnara,Mamatova,it,F,29,2024-03-01,4.5
3,103,Gulnara,Mamatova,IT,F,29,2024-03-01,4.5
4,105,Nurlan,Abdyldaev,Finance,M,150,2021-11-20,4.0


In [25]:
# how='all': строка удаляется, только если пропущены все значения
print(df.dropna(how="all").shape)

(15, 9)


In [26]:
# subset: пропуски проверяются только в указанных столбцах
print(df.dropna(subset=["Salary"]).shape)
print(df.dropna(subset=["Age"]).shape)

(12, 9)
(15, 9)


In [27]:
# thresh: строка остаётся, если в ней не меньше thresh заполненных значений
print(df.dropna(thresh=9).shape)   # нужны все 9 значений
print(df.dropna(thresh=8).shape)   # достаточно 8 значений

(12, 9)
(15, 9)


### Заполнение пропусков — `fillna()`

In [28]:
# Сравниваем среднее и медиану, чтобы выбрать способ заполнения
print("Среднее:", df["Salary"].mean())
print("Медиана:", df["Salary"].median())

Среднее: 72916.66666666667
Медиана: 71000.0


In [29]:
# Заполняем пропуски медианной зарплатой
median_salary = df["Salary"].median()
df["Salary"] = df["Salary"].fillna(median_salary)

print(df["Salary"].isnull().sum())
display(df[["EmployeeID", "FirstName", "Salary"]])

0


,EmployeeID,FirstName,Salary
0,101,Ainura,80000.0
1,102,Bakyt,65000.0
2,103,Gulnara,71000.0
3,103,Gulnara,71000.0
4,105,Nurlan,90000.0
5,106,Aigul,70000.0
6,107,Beksultan,71000.0
7,108,Aisuluu,62000.0
8,109,Ermek,85000.0
9,110,Nurgul,58000.0


### Задание 2 — Пояснение

- До очистки `isnull()` нашёл только **2 пропущенные зарплаты (13,33%)**. После замены `"unknown"` на `NaN`
  пропусков стало **3 (20%)**. Процент считается через `isnull().mean() * 100`: `True` считается как 1, `False` — как 0,
  поэтому среднее равно доле пропусков.
- `pd.to_numeric(..., errors="coerce")` перевёл столбец в `float64`; любой другой неверный текст тоже стал бы `NaN`,
  а не вызвал ошибку (`errors="raise"` по умолчанию остановил бы программу).

**Что меняет каждый параметр `dropna()`:**

| Вызов | Результат | Смысл |
|---|---|---|
| `dropna()` | 12 строк | по умолчанию `axis=0, how="any"`: удаляется каждая **строка**, где есть хотя бы один NaN |
| `dropna(axis=1)` | 8 столбцов | удаляется каждый **столбец** с NaN — теряется весь столбец `Salary` |
| `dropna(how="all")` | 15 строк | строка удаляется, только если в ней **все** значения NaN — таких строк нет |
| `dropna(subset=["Salary"])` | 12 строк | NaN ищется **только в указанных столбцах** |
| `dropna(subset=["Age"])` | 15 строк | в Age нет NaN, поэтому ничего не удаляется |
| `dropna(thresh=9)` | 12 строк | остаются строки, где **не меньше 9 заполненных** значений (здесь = полные строки) |
| `dropna(thresh=8)` | 15 строк | остаются строки, где не меньше 8 значений — один NaN допускается |

Параметр `inplace=True` изменил бы сам `df`, поэтому в экспериментах он не использовался.

Удаление убрало бы 20% сотрудников (или целый столбец), поэтому для такого маленького датасета оно не подходит.
Пропущенные зарплаты **заполнены медианой (71 000)**. Среднее (≈ 72 917) немного выше, потому что на него
влияют высокие зарплаты, а медиана устойчива к крайним значениям. Заполнение константой (0) исказило бы
средние значения, а `ffill()`/`bfill()` подходят, когда важен порядок наблюдений, — здесь порядок строк ничего не значит.

## Задание 3 — Стандартизация названий отделов

In [30]:
print(df["Department"].value_counts())

Department
IT                       4
HR                       4
it                       2
Finance                  2
Financing                1
Information Tecnology    1
finance                  1
Name: count, dtype: int64


In [31]:
# 1) Убираем лишние пробелы и приводим к одному регистру
df["Department"] = df["Department"].str.strip().str.lower()

# 2) Исправляем опечатку в слове с помощью str.replace()
df["Department"] = df["Department"].str.replace("tecnology", "technology")

# 3) Заменяем альтернативные названия одним официальным (как в таблице departments)
df["Department"] = df["Department"].replace({
    "it": "IT",
    "information technology": "IT",
    "hr": "HR",
    "finance": "Finance",
    "financing": "Finance",
})

print(df["Department"].unique())
print(df["Department"].value_counts())

<StringArray>
['IT', 'HR', 'Finance']
Length: 3, dtype: str


Department
IT         7
HR         4
Finance    4
Name: count, dtype: int64


In [32]:
# Находим сотрудника с неверным возрастом
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")
display(df[(df["Age"] < 18) | (df["Age"] > 70)])

,EmployeeID,FirstName,LastName,Department,Gender,Age,Salary,HireDate,PerformanceScore
4,105,Nurlan,Abdyldaev,Finance,M,150,90000.0,2021-11-20,4.0


In [33]:
# Решение: 150 лет невозможно, а настоящий возраст угадать нельзя (15? 50?),
# поэтому значение заменяется на пропуск, а не на придуманное число.
df.loc[(df["Age"] < 18) | (df["Age"] > 70), "Age"] = np.nan
display(df[df["EmployeeID"] == 105][["EmployeeID", "FirstName", "LastName", "Age"]])

,EmployeeID,FirstName,LastName,Age
4,105,Nurlan,Abdyldaev,NaN


### Задание 3 — Пояснение

- В столбце `Department` было **7 разных написаний** для 3 реальных отделов. Порядок очистки:
  `str.strip()` убирает пробелы → `str.lower()` приводит к одному регистру → `str.replace()` исправляет опечатку
  `"tecnology"` → `replace({...})` заменяет альтернативные названия (`"information technology"`, `"financing"`)
  и приводит значения к официальным названиям из таблицы `departments` (`IT`, `HR`, `Finance`).
- После очистки осталось **только 3 уникальных значения**: IT (7), HR (4), Finance (4).
- **Неверный возраст:** у Nurlan Abdyldaev (ID 105) `Age = 150`, что невозможно. Допустимый диапазон — 18–70 лет.
  Я **не стала** заменять его медианой, потому что это было бы придуманное значение для реального человека; через
  `df.loc[...]` возраст заменён на пропуск (`NaN`), чтобы HR мог уточнить его у сотрудника. Остальные данные этого
  сотрудника верные, поэтому строка сохранена. Теперь Age — число (`float64` из-за NaN); в задании 5 он станет
  nullable integer.

## Задание 4 — Удаление дубликатов

In [34]:
# В исходных данных НЕТ полных дубликатов, потому что "it"/"IT" и "F"/"f" различались
print("В исходных данных:", employees.duplicated().sum())
print("После очистки:    ", df.duplicated().sum())

В исходных данных: 0
После очистки:     1


In [35]:
# Просмотр дубликатов
display(df[df.duplicated(keep=False)])

,EmployeeID,FirstName,LastName,Department,Gender,Age,Salary,HireDate,PerformanceScore
2,103,Gulnara,Mamatova,IT,F,29.0,71000.0,2024-03-01,4.5
3,103,Gulnara,Mamatova,IT,F,29.0,71000.0,2024-03-01,4.5


### Эксперименты с `drop_duplicates()` (результат не сохраняется в `df`)

In [36]:
# keep="first" (по умолчанию) — остаётся первая запись (индекс 2)
kept_first = df.drop_duplicates(subset=["EmployeeID"], keep="first")
display(kept_first[kept_first["EmployeeID"] == 103])

,EmployeeID,FirstName,LastName,Department,Gender,Age,Salary,HireDate,PerformanceScore
2,103,Gulnara,Mamatova,IT,F,29.0,71000.0,2024-03-01,4.5


In [37]:
# keep="last" — остаётся последняя запись (индекс 3)
kept_last = df.drop_duplicates(subset=["EmployeeID"], keep="last")
display(kept_last[kept_last["EmployeeID"] == 103])

,EmployeeID,FirstName,LastName,Department,Gender,Age,Salary,HireDate,PerformanceScore
3,103,Gulnara,Mamatova,IT,F,29.0,71000.0,2024-03-01,4.5


In [38]:
# keep=False — удаляются ВСЕ копии, сотрудник 103 исчезает полностью
print(df.drop_duplicates(subset=["EmployeeID"], keep=False).shape)
print(df.drop_duplicates(subset=["EmployeeID"], keep=False)["EmployeeID"].tolist())

(13, 9)
[101, 102, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115]


In [39]:
# Шаг 1: удаляем полные дубликаты
df = df.drop_duplicates()
print(df.shape)

# Шаг 2: удаляем дубликаты по EmployeeID; ignore_index=True заново нумерует строки 0, 1, 2, ...
df = df.drop_duplicates(subset=["EmployeeID"], keep="first", ignore_index=True)
print(df.shape)
print(df["EmployeeID"].duplicated().sum())

(14, 9)
(14, 9)
0


### Задание 4 — Пояснение

- В **исходных** данных `duplicated()` находит **0** полных дубликатов, потому что две записи ID 103 написаны
  по-разному (`"it"` и `"IT"`, `"F"` и `"f"`). После очистки в заданиях 1–3 они стали одинаковыми, и `duplicated()`
  находит **1** полный дубликат. Это показывает, что очистку нужно делать до удаления дубликатов.
- `drop_duplicates()` без параметров сравнивает **все столбцы**; `subset=["EmployeeID"]` сравнивает только ID и
  находит записи одного сотрудника, даже если они отличаются в других столбцах.
- `keep="first"` оставляет первое вхождение (индекс 2), `keep="last"` — последнее (индекс 3),
  а `keep=False` удаляет **все** копии — сотрудник 103 исчез бы полностью (13 строк), что неверно для подсчёта
  сотрудников.
- Итог: удалены полные дубликаты (15 → 14 строк), затем дубликаты по `EmployeeID` с `keep="first"` (больше не
  осталось, 14 строк). `ignore_index=True` заново пронумеровал строки. Теперь каждый `EmployeeID` уникален, и
  количество сотрудников верное: **14 человек**.

## Задание 5 — Преобразование типов данных

In [40]:
print(df.dtypes)

EmployeeID            int64
FirstName               str
LastName                str
Department              str
Gender                  str
Age                 float64
Salary              float64
HireDate                str
PerformanceScore     object
dtype: object


In [41]:
# Age -> nullable integer. Обычный astype(int) не работает, если в столбце есть NaN,
# а "Int64" (с заглавной I) умеет хранить пропуск как <NA>
df["Age"] = df["Age"].astype("Int64")
print(df["Age"].dtype)
print(df["Age"].tolist())

Int64
[28, 35, 29, <NA>, 31, 33, 27, 30, 38, 26, 34, 29, 32, 31]


In [42]:
# Память ДО перевода в category (последняя строка вывода — memory usage)
df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   EmployeeID        14 non-null     int64  
 1   FirstName         14 non-null     str    
 2   LastName          14 non-null     str    
 3   Department        14 non-null     str    
 4   Gender            14 non-null     str    
 5   Age               13 non-null     Int64  
 6   Salary            14 non-null     float64
 7   HireDate          14 non-null     str    
 8   PerformanceScore  14 non-null     object 
dtypes: Int64(1), float64(1), int64(1), object(1), str(5)
memory usage: 4.7 KB


In [43]:
# Department -> category
df["Department"] = df["Department"].astype("category")
print(df["Department"].dtype)
print(df["Department"].cat.categories)

category
Index(['Finance', 'HR', 'IT'], dtype='str')


In [44]:
# Gender -> category
df["Gender"] = df["Gender"].astype("category")
print(df["Gender"].dtype)
print(df["Gender"].cat.categories)

category
Index(['F', 'M'], dtype='str')


In [45]:
# PerformanceScore -> число (текст "4.0" становится 4.0)
df["PerformanceScore"] = pd.to_numeric(df["PerformanceScore"], errors="coerce")
print(df["PerformanceScore"].dtype)
print(df["PerformanceScore"].isnull().sum())

float64


0


In [46]:
# Память и типы ПОСЛЕ преобразований
df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   EmployeeID        14 non-null     int64   
 1   FirstName         14 non-null     str     
 2   LastName          14 non-null     str     
 3   Department        14 non-null     category
 4   Gender            14 non-null     category
 5   Age               13 non-null     Int64   
 6   Salary            14 non-null     float64 
 7   HireDate          14 non-null     str     
 8   PerformanceScore  14 non-null     float64 
dtypes: Int64(1), category(2), float64(2), int64(1), str(3)
memory usage: 3.2 KB


### Задание 5 — Пояснение

- **`Age` → `Int64`** (nullable integer, с заглавной *I*). Обычный `astype(int)` выдаёт ошибку, если в столбце есть
  `NaN`, поэтому столбец остался бы `float64` (29.0, 150.0…). `Int64` хранит целые числа и показывает пропущенный
  возраст ID 105 как `<NA>`.
- **`Department`, `Gender` → `category`**, **`PerformanceScore` → `float64`** (текст `"4.0"` стал числом 4.0,
  новых пропусков не появилось). После каждого преобразования тип проверен через `dtype`.

**`object` и `category` — в чём разница:**

| `object` (текст) | `category` |
|---|---|
| каждая ячейка хранит свою полную строку | уникальные значения хранятся один раз (`categories`), а в ячейке — только маленький числовой код |
| можно записать любое значение, в том числе с опечаткой | допускаются только заданные категории |
| больше памяти, медленнее группировка | меньше памяти и быстрее группировка — лучше всего для столбцов с **небольшим числом повторяющихся значений**, как Department или Gender |
| подходит для уникального текста (имена) | бесполезен для уникальных значений, например `FirstName` |

(В pandas 3 текстовые столбцы показываются как `str` вместо `object`, но разница с `category` та же.)
Сравнение `info(memory_usage="deep")` до и после показывает, что датасет стал занимать меньше памяти (≈ 4.7 KB → 3.2 KB); на большом
HR-датасете с тысячами сотрудников экономия будет намного больше.

## Задание 6 — Разбор дат найма

In [47]:
print(df["HireDate"].dtype)

# Переводим текст в дату; errors="coerce" превратил бы неверную дату в NaT
df["HireDate"] = pd.to_datetime(df["HireDate"], format="%Y-%m-%d", errors="coerce")

print(df["HireDate"].dtype)
print(df["HireDate"].isnull().sum())

str


datetime64[us]
0


In [48]:
# Извлекаем компоненты даты
df["HireYear"] = df["HireDate"].dt.year
df["HireMonth"] = df["HireDate"].dt.month
df["HireWeekday"] = df["HireDate"].dt.day_name()

# Переводим названия дней недели на русский
df["HireWeekday"] = df["HireWeekday"].replace({
    "Monday": "понедельник", "Tuesday": "вторник", "Wednesday": "среда", "Thursday": "четверг",
    "Friday": "пятница", "Saturday": "суббота", "Sunday": "воскресенье",
})

display(df[["EmployeeID", "FirstName", "HireDate", "HireYear", "HireMonth", "HireWeekday"]])

,EmployeeID,FirstName,HireDate,HireYear,HireMonth,HireWeekday
0,101,Ainura,2023-01-15,2023,1,воскресенье
1,102,Bakyt,2022-06-10,2022,6,пятница
2,103,Gulnara,2024-03-01,2024,3,пятница
3,105,Nurlan,2021-11-20,2021,11,суббота
4,106,Aigul,2023-08-05,2023,8,суббота
5,107,Beksultan,2020-05-12,2020,5,вторник
6,108,Aisuluu,2022-09-18,2022,9,воскресенье
7,109,Ermek,2021-03-25,2021,3,четверг
8,110,Nurgul,2019-07-14,2019,7,воскресенье
9,111,Daniyar,2023-04-18,2023,4,вторник


In [49]:
# Стаж в компании (в годах) на сегодняшнюю дату
today = pd.Timestamp.today().normalize()
print("Дата расчёта:", today.date())

df["YearsWorked"] = ((today - df["HireDate"]).dt.days / 365.25).round(1)
display(df[["EmployeeID", "FirstName", "HireDate", "YearsWorked"]])

Дата расчёта: 2026-10-23


,EmployeeID,FirstName,HireDate,YearsWorked
0,101,Ainura,2023-01-15,3.8
1,102,Bakyt,2022-06-10,4.4
2,103,Gulnara,2024-03-01,2.6
3,105,Nurlan,2021-11-20,4.9
4,106,Aigul,2023-08-05,3.2
5,107,Beksultan,2020-05-12,6.4
6,108,Aisuluu,2022-09-18,4.1
7,109,Ermek,2021-03-25,5.6
8,110,Nurgul,2019-07-14,7.3
9,111,Daniyar,2023-04-18,3.5


In [50]:
# Пример анализа по времени: сколько сотрудников нанято в каждом году
print(df["HireYear"].value_counts().sort_index())

HireYear
2019    1
2020    2
2021    3
2022    3
2023    4
2024    1
Name: count, dtype: int64


### Задание 6 — Пояснение

- `pd.to_datetime()` с `format="%Y-%m-%d"` перевёл `HireDate` из текста в **`datetime64`**. Точный формат гарантирует,
  что день и месяц не перепутаются; `errors="coerce"` превратил бы неверную дату в `NaT` (все 14 дат оказались
  корректными).
- С помощью `.dt` извлечены **`HireYear`** (`.dt.year`), **`HireMonth`** (`.dt.month`) и **`HireWeekday`**
  (`.dt.day_name()`; названия дней переведены на русский через `replace()`).
- **`YearsWorked`** (стаж) = (сегодня − HireDate) в днях ÷ 365,25 (средняя длина года с учётом високосных),
  округлено до одного знака. Так как расчёт идёт от сегодняшней даты, при повторном запуске значения немного растут.
- Теперь можно отвечать на вопросы по времени: например, сколько людей наняли в каждом году или кто работает
  дольше всех (Nurgul Jumabaeva, нанята в 2019 году).

## Задание 7 — Переименование столбцов и объединение датасетов

In [51]:
df = df.rename(columns={
    "Salary": "MonthlySalary",
    "HireDate": "EmploymentStartDate"
})
print(df.columns)

Index(['EmployeeID', 'FirstName', 'LastName', 'Department', 'Gender', 'Age',
       'MonthlySalary', 'EmploymentStartDate', 'PerformanceScore', 'HireYear',
       'HireMonth', 'HireWeekday', 'YearsWorked'],
      dtype='str')


In [52]:
# Left join: сохраняем ВСЕХ сотрудников и добавляем информацию об отделе.
# Ключ называется по-разному, поэтому используем left_on и right_on.
hr = pd.merge(
    df,
    departments,
    left_on="Department",
    right_on="DepartmentName",
    how="left"
)

print(df.shape)
print(hr.shape)

(14, 13)
(14, 16)


In [53]:
# Проверяем, появились ли пропуски после объединения
print(hr[["DepartmentID", "DepartmentName", "Location"]].isnull().sum())

DepartmentID      0
DepartmentName    0
Location          0
dtype: int64


In [54]:
# Для сравнения: то же объединение на ИСХОДНЫХ (неочищенных) названиях отделов
raw_merge = pd.merge(employees, departments, left_on="Department", right_on="DepartmentName", how="left")
print(raw_merge["DepartmentID"].isnull().sum())
display(raw_merge[raw_merge["DepartmentID"].isnull()][["EmployeeID", "Department", "DepartmentID", "Location"]])

6


,EmployeeID,Department,DepartmentID,Location
2,103,it,NaN,NaN
6,107,IT,NaN,NaN
10,111,it,NaN,NaN
11,112,Financing,NaN,NaN
12,113,Information Tecnology,NaN,NaN
14,115,finance,NaN,NaN


In [55]:
# Убираем повторяющийся столбец DepartmentName и возвращаем Department тип category
hr = hr.drop(columns=["DepartmentName"])
hr["DepartmentID"] = hr["DepartmentID"].astype("Int64")
hr["Department"] = hr["Department"].astype("category")

# Меняем порядок столбцов: идентификаторы в начале, связанные столбцы рядом
new_order = [
    "EmployeeID", "FirstName", "LastName", "Gender", "Age",
    "DepartmentID", "Department", "Location",
    "MonthlySalary", "PerformanceScore",
    "EmploymentStartDate", "HireYear", "HireMonth", "HireWeekday", "YearsWorked",
]
hr = hr[new_order]
display(hr)

,EmployeeID,FirstName,LastName,Gender,Age,DepartmentID,Department,Location,MonthlySalary,PerformanceScore,EmploymentStartDate,HireYear,HireMonth,HireWeekday,YearsWorked
0,101,Ainura,Asanova,F,28,1,IT,Bishkek,80000.0,4.2,2023-01-15,2023,1,воскресенье,3.8
1,102,Bakyt,Kyzylov,M,35,2,HR,Osh,65000.0,3.8,2022-06-10,2022,6,пятница,4.4
2,103,Gulnara,Mamatova,F,29,1,IT,Bishkek,71000.0,4.5,2024-03-01,2024,3,пятница,2.6
3,105,Nurlan,Abdyldaev,M,<NA>,3,Finance,Bishkek,90000.0,4.0,2021-11-20,2021,11,суббота,4.9
4,106,Aigul,Sultanova,F,31,2,HR,Osh,70000.0,3.9,2023-08-05,2023,8,суббота,3.2
5,107,Beksultan,Toktogulov,M,33,1,IT,Bishkek,71000.0,4.1,2020-05-12,2020,5,вторник,6.4
6,108,Aisuluu,Baisalova,F,27,3,Finance,Bishkek,62000.0,3.7,2022-09-18,2022,9,воскресенье,4.1
7,109,Ermek,Kadyrov,M,30,1,IT,Bishkek,85000.0,4.3,2021-03-25,2021,3,четверг,5.6
8,110,Nurgul,Jumabaeva,F,38,2,HR,Osh,58000.0,3.6,2019-07-14,2019,7,воскресенье,7.3
9,111,Daniyar,Turgunov,M,26,1,IT,Bishkek,72000.0,4.4,2023-04-18,2023,4,вторник,3.5


### Задание 7 — Пояснение

- `rename(columns={...})` переименовал `Salary` → **`MonthlySalary`** и `HireDate` → **`EmploymentStartDate`**, чтобы
  смысл столбцов был понятен.
- Таблицы объединены через `pd.merge(..., left_on="Department", right_on="DepartmentName", how="left")`.
  `left_on`/`right_on` нужны, потому что ключевой столбец в таблицах называется по-разному.
  `how="left"` сохраняет **всех 14 сотрудников**, даже если отдел не найден; количество строк до и после
  объединения одинаковое (14).
- **Объединение не создало пропусков** — у каждого сотрудника есть `DepartmentID` и `Location`.
  Для сравнения: при объединении **исходных** данных **6 сотрудников остаются без отдела** (`"it"`, `" IT "`,
  `"Financing"`, `"Information Tecnology"`, `"finance"`) — это доказывает, зачем было нужно задание 3.
- Повторяющийся ключ `DepartmentName` удалён, `Department` снова переведён в `category` (merge возвращает ключевой
  столбец как текст), а столбцы переставлены через список `new_order`: сначала идентификаторы, связанные столбцы — рядом.

## Задание 8 — Финальная проверка и выводы

In [56]:
# Количество строк и столбцов, типы данных и заполненные значения
print(hr.shape)
hr.info()

(14, 15)
<class 'pandas.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   EmployeeID           14 non-null     int64         
 1   FirstName            14 non-null     str           
 2   LastName             14 non-null     str           
 3   Gender               14 non-null     category      
 4   Age                  13 non-null     Int64         
 5   DepartmentID         14 non-null     Int64         
 6   Department           14 non-null     category      
 7   Location             14 non-null     str           
 8   MonthlySalary        14 non-null     float64       
 9   PerformanceScore     14 non-null     float64       
 10  EmploymentStartDate  14 non-null     datetime64[us]
 11  HireYear             14 non-null     int32         
 12  HireMonth            14 non-null     int32         
 13  HireWeekday          14 non-null     st

In [57]:
# Пропуски
print(hr.isnull().sum())

EmployeeID             0
FirstName              0
LastName               0
Gender                 0
Age                    1
DepartmentID           0
Department             0
Location               0
MonthlySalary          0
PerformanceScore       0
EmploymentStartDate    0
HireYear               0
HireMonth              0
HireWeekday            0
YearsWorked            0
dtype: int64


In [58]:
# Дубликаты
print("Полных дубликатов:", hr.duplicated().sum())
print("Дубликатов EmployeeID:", hr["EmployeeID"].duplicated().sum())

Полных дубликатов:

 0
Дубликатов EmployeeID: 0


In [59]:
# Согласованность значений и тип дат
print(hr["Department"].unique())
print(hr["Gender"].unique())
print(hr["EmploymentStartDate"].dtype)

['IT', 'HR', 'Finance']
Categories (3, str): ['Finance', 'HR', 'IT']
['F', 'M']
Categories (2, str): ['F', 'M']
datetime64[us]


In [60]:
# Описательная статистика числовых столбцов чистых данных
display(hr.describe())

,EmployeeID,Age,DepartmentID,MonthlySalary,PerformanceScore,EmploymentStartDate,HireYear,HireMonth,YearsWorked
count,14.000000,13.0,14.0,14.000000,14.000000,14,14.000000,14.000000,14.000000
mean,108.285714,31.0,1.857143,72642.857143,4.028571,2022-03-21 20:34:17.142857,2021.714286,6.642857,4.585714
min,101.000000,26.0,1.0,58000.000000,3.600000,2019-07-14 00:00:00,2019.000000,1.000000,2.600000
25%,105.250000,29.0,1.0,65250.000000,3.825000,2021-05-03 12:00:00,2021.000000,4.250000,3.575000
50%,108.500000,31.0,2.0,71000.000000,4.000000,2022-07-30 00:00:00,2022.000000,6.500000,4.250000
75%,111.750000,33.0,2.75,79500.000000,4.200000,2023-03-25 18:00:00,2023.000000,8.750000,5.475000
max,115.000000,38.0,3.0,90000.000000,4.500000,2024-03-01 00:00:00,2024.000000,12.000000,7.300000
std,4.496641,3.391165,0.864438,10195.075303,0.267261,NaN,1.437336,3.248837,1.369928


In [61]:
# Исходный датасет не изменён
print(employees.shape)
display(employees.head())

(15, 9)


,EmployeeID,FirstName,LastName,Department,Gender,Age,Salary,HireDate,PerformanceScore
0,101,Ainura,Asanova,IT,F,28,80000,2023-01-15,4.2
1,102,Bakyt,Kyzylov,HR,m,35,65000,2022-06-10,3.8
2,103,Gulnara,Mamatova,it,F,29,NaN,2024-03-01,4.5
3,103,Gulnara,Mamatova,IT,f,29,NaN,2024-03-01,4.5
4,105,Nurlan,Abdyldaev,Finance,M,150,90000,2021-11-20,4.0


### Задание 8 — Выводы

**Итоговый результат:** чистая таблица `hr` — **14 уникальных сотрудников и 15 столбцов**. Все названия отделов
согласованы (IT, HR, Finance) и сопоставлены с таблицей `departments`, `EmploymentStartDate` имеет тип `datetime64`,
`MonthlySalary` и `PerformanceScore` — числовые, `Department`/`Gender` — категории, дубликатов нет.
Единственный оставшийся пропуск — **возраст сотрудника 105**, он оставлен как `<NA>` намеренно, потому что
настоящее значение неизвестно. Исходные датасеты `employees` и `departments` не изменены.

**Найденные проблемы и способы их решения:**

| Проблема | Метод и параметры | Зачем |
|---|---|---|
| Лишние пробелы (`" IT "`, `"Jumabaeva "`) | `str.strip()` | из-за пробелов одинаковые значения выглядят разными |
| Разный регистр (`m`/`f`, `MOLDOKULOV`) | `str.upper()`, `str.title()`, `str.lower()` | одно написание для одного значения |
| `"unknown"` и NaN в Salary (20%) | `replace("unknown", np.nan)`, `pd.to_numeric(errors="coerce")`, `fillna(медиана)` | скрытые пропуски; медиана устойчива к крайним значениям и сохраняет все строки |
| Несогласованные отделы (7 написаний) | `str.lower()`, `str.replace()`, `replace({...})` | нужно для правильной группировки и объединения |
| Невозможный возраст 150 | условие `(Age < 18) \| (Age > 70)`, `df.loc[...] = np.nan` | придуманное число хуже, чем известный пропуск |
| Дубликат сотрудника 103 | `drop_duplicates()`, затем `subset=["EmployeeID"], keep="first", ignore_index=True` | правильное количество сотрудников |
| Неверные типы | `astype("Int64")`, `astype("category")`, `pd.to_numeric()` | вычисления, экономия памяти, пропуски в целых числах |
| Даты в виде текста | `pd.to_datetime(format="%Y-%m-%d")`, `.dt.year`, `.dt.month`, `.dt.day_name()` | анализ по времени, расчёт стажа |
| Непонятные названия столбцов / отдельные таблицы | `rename(columns=...)`, `pd.merge(how="left", left_on, right_on)` | понятные названия, локация отдела для каждого сотрудника |

**Почему подготовка данных важна перед HR-анализом.** Без очистки анализ дал бы неверные ответы: сотрудник 103
был бы посчитан дважды, отдел IT распался бы на четыре разных отдела, средний возраст завысило бы значение 150,
зарплаты нельзя было бы усреднить, потому что они были текстом, а 6 сотрудников потеряли бы информацию об отделе
при объединении. Решения о зарплатах, найме и оценке работы хороши ровно настолько, насколько хороши данные,
на которых они основаны, — *garbage in, garbage out* («мусор на входе — мусор на выходе»).